This is an example of generating DayMet h5 files to drive ATS 2D transect simulations.

- Input
    - `data-processed/{site_name}/m2_coords_{site_name}.mat`
    - [optional] DayMet raw data, script can also download
- Output
    - `data-processed/{watershed_name}/{watershed_name}_DayMet_2013_2023.h5`
        - -> `outputs['daymet_filename_watershed']`
    - `data-processed/{site_name}/{site_name}_DayMet_2013_2023.h5`
        - -> `outputs['daymet_filename_site']`
    - `data-processed/{site_name}/{site_name}_DayMet_typical10yr_2013_2023.h5`
        - -> `outputs['daymet_spinup_filename_site']`

**File History**

update 2026/1/29
- a watershed-workflow 2.0 based notebook

update 2025/10/13
- update `config.json`. Mainly revise the model run pipeline.

update 2025/8/32
- add `config.json`

In [ ]:
%load_ext autoreload
%autoreload 2

# Parameters and data sources

In [ ]:
# Parameters cell
import json
with open('config.json', 'r') as f:
    config = json.load(f)
watershed_name = config['watershed_name']
# hucs           = [config['hucs']]
site_name      = config['site_name']

# simulation control
start_year_spinup         = config['start_year_spinup']
end_year_spinup           = config['end_year_spinup']
nyears_steadystate_spinup = config['nyears_steadystate_spinup']
nyears_cyclic_spinup      = config['nyears_cyclic_spinup']
start_year_transient      = config['start_year_transient']
end_year_transient        = config['end_year_transient']

In [ ]:
outputs={}

In [ ]:
import logging
import sys,os
sys.path.append(os.path.join(os.environ['ATS_SRC_DIR'],'tools','meshing','meshing_ats'))
# import meshing_ats
import h5py

import geopandas as gpd
import numpy as np
import pandas as pd
pd.options.display.max_columns = None
pd.options.display.max_rows = 20

import rasterio
import shapely
from shapely.geometry import Point, LineString, Polygon, box, mapping
#import fiona
import scipy.ndimage
import cftime, datetime

import matplotlib
import matplotlib.colors as colors
from matplotlib import pyplot as plt

# import watershed_workflow
# #import watershed_workflow.source_list
# import watershed_workflow.ui
# import watershed_workflow.colors
# import watershed_workflow.condition
# import watershed_workflow.mesh
# import watershed_workflow.split_hucs
# import watershed_workflow.soil_properties
# import watershed_workflow.daymet
# import watershed_workflow.utils
# import watershed_workflow.regions


# # ats_input_spec library, to be moved to amanzi_xml
# import ats_input_spec
# import ats_input_spec.public
# import ats_input_spec.io

# # amanzi_xml, included in AMANZI_SRC_DIR/tools/amanzi_xml
# import amanzi_xml.utils.io as aio
# import amanzi_xml.utils.search as asearch
# import amanzi_xml.utils.errors as aerrors
# from amanzi_xml.common.parameter import Parameter
# from amanzi_xml.common.parameter_list import ParameterList
# from ats_input_spec.public import known_specs

from scipy.io import loadmat
import h5py as h5
import xarray as xr

In [ ]:
import watershed_workflow 
import watershed_workflow.config
import watershed_workflow.sources
import watershed_workflow.utils
import watershed_workflow.plot
import watershed_workflow.mesh
import watershed_workflow.regions
import watershed_workflow.meteorology
import watershed_workflow.land_cover_properties
import watershed_workflow.resampling
import watershed_workflow.condition
import watershed_workflow.io
import watershed_workflow.sources.standard_names as names

In [ ]:
# set up a dictionary of source objects
sources = watershed_workflow.sources.getDefaultSources()
sources['hydrography'] = watershed_workflow.sources.hydrography_sources['NHDPlus HR']
#sources['HUC'] = watershed_workflow.source_list.huc_sources['NHD Plus']
#sources['DEM'] = watershed_workflow.source_list.dem_sources['NED 1/3 arc-second']
sources['geologic structure'] = watershed_workflow.sources.ManagerGLHYMPS('./data/soil_structure/GLHYMPS/GLHYMPS.shp')
sources['depth to bedrock'] = watershed_workflow.sources.ManagerRaster('./data/soil_structure/SoilGrids2017/BDTICM_M_250m_ll.tif')
watershed_workflow.sources.logSources(sources)
#sources

In [ ]:
# Note that, by default, we tend to work in the DayMet CRS because this allows us to avoid
# reprojecting meteorological forcing datasets.
crs_daymet = watershed_workflow.crs.daymet_crs
crs_latlon = watershed_workflow.crs.latlon_crs # essentially epsg(4269)
# note: epsg(4269) i.e. NAD83 vs epsg(4326) i.e. WGS84
# - https://gis.stackexchange.com/questions/170839/is-re-projection-needed-from-srid-4326-wgs-84-to-srid-4269-nad-83

# alternative
#proj_daymet = "+proj=lcc +lat_1=25 +lat_2=60 +lat_0=42.5 +lon_0=-100 +x_0=0 +y_0=0 +datum=WGS84" # daymet crs
#proj_wgs84  = "epsg:4326" # latlon
#crs_daymet  = watershed_workflow.crs.from_string(proj_daymet)
#crs_wgs84   = watershed_workflow.crs.from_string(proj_wgs84)

# Prepare watershed shape and hillslope shape

In [ ]:
# load from watershed shp
#watershed_name = 'OakCreek' # name the domain, used in filenames, etc
fname_watershed_shp = f'../data-processed/{watershed_name}/{watershed_name}_bounds.shp'
watershed_shape = gpd.read_file(fname_watershed_shp)

In [ ]:
# load hillslope geometry from mat file generated in "1-full_workflow_OakCreek.ipynb"
#site_name = 'NF01'
#meshsize_nx = 100

m2_mat_filename =  f'../data-processed/{site_name}/m2_coords_{site_name}.mat'
loaded_data = loadmat(m2_mat_filename)
meshsize_nx = loaded_data['meshsize_nx'].flatten()[0]
#dzs_soil  = loaded_data['dzs_soil'].flatten()
#dzs_geo   = loaded_data['dzs_geo'].flatten()
#m2_coords = loaded_data['m2_coords']
loaded_gdf_dict = loaded_data['gdf_data']
gdf_reloaded = pd.DataFrame({
    'lon': loaded_gdf_dict['lon'][0, 0].flatten(),
    'lat': loaded_gdf_dict['lat'][0, 0].flatten(),
    'h_distance': loaded_gdf_dict['h_distance'][0, 0].flatten(),
    'elevation': loaded_gdf_dict['elevation'][0, 0].flatten()
})
geometry = [Point(xy) for xy in zip(gdf_reloaded['lon'], gdf_reloaded['lat'])]
hillslope_gdf = gpd.GeoDataFrame(gdf_reloaded, geometry=geometry)

# create hillslope polygon - already a shapely object, no conversion needed
xsec_plg = Polygon([hillslope_gdf.geometry[i] for i in range(hillslope_gdf.shape[0])])
xsec_plg_dict = {"type": "Feature", "id":0, "properties":{}, "geometry": mapping(xsec_plg)}
# In watershed-workflow 2.0, use the shapely Polygon directly
xsec_plg_dict_shply = xsec_plg

# convert to latlon crs using warp.shply() in v2.0
# reproj_xsec_plg = watershed_workflow.warp.shply(xsec_plg, crs_daymet, crs_latlon)

In [ ]:
hillslope_gdf

# Get DayMet data downloaded for the watershed

originally copied from `OakCreek_ATS_3D/notebooks/data/meterology/daymet` to `OakCreek_ATS_2D/notebooks/data/meteorology/daymet`

In [ ]:
generate_daymet=True

#start_year = 2013 #1980
#end_year = 2023
#nyears_cyclic_steadystate = 10

In [ ]:
# Check the actual CRS of the watershed shapefile
print(f"Watershed shapefile CRS: {watershed_shape.crs}")
print(f"DayMet CRS: {crs_daymet}")
print(f"Watershed bounds: {watershed_shape.total_bounds}")

In [ ]:
# outputs['daymet_filename_watershed'] = f'../data-processed/{watershed_name}/{watershed_name}_DayMet_{start_year_spinup}_{end_year_transient}.h5'

if generate_daymet:
    startdate_spinup    = f"{start_year_spinup}-1-1"
    enddate_spinup      = f"{end_year_spinup+1}-1-1"
    startdate_transient = f"{start_year_transient}-1-1"
    enddate_transient   = f"{end_year_transient+1}-1-1" # tested for ww1.5
    
    # Use the watershed polygon geometry directly (not just bounds)
    # This ensures proper CRS handling through the entire workflow
    watershed_polygon = watershed_shape.geometry.union_all() if hasattr(watershed_shape.geometry, 'union_all') else watershed_shape.geometry.iloc[0]
    
    # In v2.0, geopandas CRS is already a pyproj CRS object - use directly
    watershed_crs = watershed_shape.crs
    
#     source = watershed_workflow.sources.manager_daymet.FileManagerDaymet()
#     met_data = source.get_data(watershed_polygon, watershed_crs, startdate_spinup, enddate_transient)

#     # watershed-workflow v1.5
#     # unit conversion is included here, for prcp, mm/day -> m/s
#     met_data_ats = watershed_workflow.daymet.convertToATS(met_data)
#     bounds = tuple(watershed_shape.total_bounds)  # Keep for attrs
#     attrs = watershed_workflow.daymet.getAttributes(bounds, startdate_spinup, enddate_transient)
#     watershed_workflow.io.write_dataset_to_hdf5(outputs['daymet_filename_watershed'], met_data_ats, attrs)

In [ ]:
#met_data_raw = sources['meteorology'].getDataset(watershed.exterior, crs, start_leap, end_leap)
met_data_raw = sources['meteorology'].getDataset(watershed_polygon, watershed_crs, startdate_spinup, enddate_transient)

In [ ]:
# convert it to daily mean immediately
met_data_raw_daily = met_data_raw.resample(time=datetime.timedelta(hours=24)).mean()

In [ ]:
# also, note that we sampled from 8/1 to 8/1, meaning we have an extra day now.
met_data_raw_daily = met_data_raw_daily.isel({'time' : slice(0, -1)})
print(f'Total days: {len(met_data_raw_daily['time'])}')

In [ ]:
met_data_raw_daily

In [ ]:
# AORC is a non-projected dataset in lat-lon
# warp it to projected
met_data_warped = watershed_workflow.warp.dataset(met_data_raw_daily, crs_daymet, 'bilinear')
met_data_warped_noleap = watershed_workflow.data.filterLeapDay(met_data_warped)


# convert and write ATS format for transient run
met_data_transient_noleap = watershed_workflow.meteorology.convertAORCToATS(met_data_warped_noleap)
met_data_transient_noleap

In [ ]:
# plot a few of the met data -- does it look reasonable?
def plotMetData(met, x=60, y=60):
    """plot one pixel as a function of time"""
    fig = plt.figure()
    ax = fig.add_subplot(121)
    
    met_data_single_pixel = met.isel({'time':slice(0,365),
                                               'x' : x,
                                               'y' : y})
    
    met_data_single_pixel['precipitation rain [m s^-1]'].plot(color='b', label='rain')
    met_data_single_pixel['precipitation snow [m SWE s^-1]'].plot(color='c', label='snow')
    ax.set_ylabel('precip [m s^-1]')
    ax.set_title('')
    ax.legend()
    
    ax = fig.add_subplot(122)
    met_data_single_pixel['incoming shortwave radiation [W m^-2]'].plot(color='r', label='qSW_in')
    ax.set_ylabel('incoming shortwave radiation [W m^-2]')
    ax.set_title('')
    
    plt.show()

plotMetData(met_data_transient_noleap)

In [ ]:
# import xarray as xr
# temp_nc_daymet_dayl = xr.open_dataset("./data/meteorology/daymet/daymet_prcp_1980_46.7991x-121.0852_46.6705x-120.8034.nc")
# print(temp_nc_daymet_dayl)

Unit of Daymet vars - https://daymet.ornl.gov/overview

| Parameter	| Abbr	| Units	| Description |
| ---- | ---- | ---- | ---- |
|Day length	|dayl	|s/day	|Duration of the daylight period in seconds per day. This calculation is based on the period of the day during which the sun is above a hypothetical flat horizon|
|Precipitation	|prcp	|mm/day	|Daily total precipitation in millimeters per day, sum of all forms converted to water-equivalent. Precipitation occurrence on any given day may be ascertained.|
|Shortwave radiation	|srad	|W/m2	|Incident shortwave radiation flux density in watts per square meter, taken as an average over the daylight period of the day. NOTE: Daily total radiation (MJ/m2/day) can be calculated as follows: ((srad (W/m2) * dayl (s/day)) / l,000,000)|
|Snow water equivalent	|swe	|kg/m2	|Snow water equivalent in kilograms per square meter. The amount of water contained within the snowpack.|
|Maximum air temperature	|tmax	|degrees C	|Daily maximum 2-meter air temperature in degrees Celsius.|
|Minimum air temperature	|tmin	|degrees C	|Daily minimum 2-meter air temperature in degrees Celsius.|
|Water vapor pressure	|vp	|Pa	|Water vapor pressure in pascals. Daily average partial pressure of water vapor.|

In [ ]:
## watershed-workflow 2.0 uses aorc instead of daymet

In [ ]:
met_data_warped_noleap

In [ ]:
# List all available data variables
print("Available data variables:")
print(list(met_data_warped_noleap.data_vars))

In [ ]:
# Quick view of all variables and their attributes
for var_name in met_data_warped_noleap.data_vars:
    var = met_data_warped_noleap[var_name]
    long_name = var.attrs.get('long_name', 'N/A')
    units = var.attrs.get('units', 'N/A')
    print(f"{var_name}: {long_name} [{units}]")

## Plot DayMet with 2D transect

In [ ]:
m2_mat_filename =  f'../data-processed/{site_name}/startendcoords_{site_name}.mat'
loaded_data  = loadmat(m2_mat_filename)
start_coords = loaded_data['start_coords'].flatten()
end_coords   = loaded_data['end_coords'].flatten()

print(start_coords)
print(end_coords)

In [ ]:
# control xlim and ylim of plot
dx = 5000/2
dy = 4000/2
xmin = (start_coords[0]+end_coords[0])/2 - dx/2
xmax = (start_coords[0]+end_coords[0])/2 + dx/2
ymin = (start_coords[1]+end_coords[1])/2 - dy/2
ymax = (start_coords[1]+end_coords[1])/2 + dy/2

print([xmin, xmax, ymin, ymax])

In [ ]:
# Plot the datasets using standard matplotlib and cartopy
# ['APCP_surface', 'DLWRF_surface', 'DSWRF_surface', 'PRES_surface', 'SPFH_2maboveground', 'TMP_2maboveground', 'UGRD_10maboveground', 'VGRD_10maboveground']
ivar = 'APCP_surface'
islice = 100

import cartopy.crs as ccrs

# Get the cartopy CRS using the proj4 string
# DayMet uses Lambert Conformal Conic projection
crs_daymet_cartopy = ccrs.LambertConformal(
    central_longitude=-100,
    central_latitude=42.5,
    standard_parallels=(25, 60),
    globe=ccrs.Globe(ellipse='WGS84')
)

# Create figure with cartopy subplots
fig = plt.figure(figsize=(15, 6))
ax1 = fig.add_subplot(1, 2, 1, projection=crs_daymet_cartopy)
ax2 = fig.add_subplot(1, 2, 2, projection=crs_daymet_cartopy)

# Get the raster data from xarray
raster_data = met_data_warped_noleap[ivar].isel(time=islice).values

# Get spatial extent from xarray coordinates
x_coords = met_data_warped_noleap.x.values
y_coords = met_data_warped_noleap.y.values

# Calculate extent [left, right, bottom, top]
# Assuming coordinates are at cell centers, expand by half a cell
dx = x_coords[1] - x_coords[0] if len(x_coords) > 1 else 1
dy = y_coords[1] - y_coords[0] if len(y_coords) > 1 else 1

left = x_coords[0] - dx/2
right = x_coords[-1] + dx/2
top = y_coords[0] - dy/2
bottom = y_coords[-1] + dy/2

extent = [left, right, bottom, top]
height, width = raster_data.shape

# Plot raster on ax1
im1 = ax1.imshow(raster_data, extent=extent, origin='upper', 
                 transform=crs_daymet_cartopy, cmap='viridis')
plt.colorbar(im1, ax=ax1, label=f'{ivar}')

# Plot shapefiles on ax1
# Convert shapely polygon to plot
if hasattr(xsec_plg_dict_shply, 'exterior'):
    x, y = xsec_plg_dict_shply.exterior.xy
    ax1.plot(x, y, color='r', linewidth=2, transform=crs_daymet_cartopy, label='Hillslope')

# Plot watershed boundary
for geom in watershed_shape.geometry:
    if hasattr(geom, 'exterior'):
        x, y = geom.exterior.xy
        ax1.plot(x, y, color='orange', linewidth=2, transform=crs_daymet_cartopy, label='Watershed')
    
ax1.set_title("Raw meteorology")
ax1.legend()

# Plot raster on ax2 (zoomed)
im2 = ax2.imshow(raster_data, extent=extent, origin='upper',
                 transform=crs_daymet_cartopy, cmap='viridis')
plt.colorbar(im2, ax=ax2, label=f'{ivar}')

# Plot shapefile on ax2
if hasattr(xsec_plg_dict_shply, 'exterior'):
    x, y = xsec_plg_dict_shply.exterior.xy
    ax2.plot(x, y, color='r', linewidth=2, transform=crs_daymet_cartopy, label='Hillslope')

ax2.set_title("Zoom in")
ax2.set_xlim(xmin, xmax)
ax2.set_ylim(ymin, ymax)
ax2.legend()

plt.tight_layout()
plt.show()

# Get Daymet for the hillslope site

modified from Bing's "2a-get_daymet.ipynb". First, extract 2D data. Then, generate typical year data for spinup.

## Find value at x and y in 2D transect

Use bilinear interpolation to get Daymet forcing at each location of the 2D transect

In [ ]:
from scipy import interpolate
from tqdm import tqdm
def bilinear_interpolate(x1,y1,x2,y2,dat1):
    dat2 = {}
    coord1 = np.concatenate([[x1.flatten()], [y1.flatten()]]).T
    coord2 = np.concatenate([[x2.flatten()], [y2.flatten()]]).T
    for varn, d1 in dat1.items():
        d1[d1==-9999] = np.nan 
        ntime, _, _ = d1.shape
        d2 = np.zeros([ntime, 2, len(x2)])
        for i in tqdm(range(ntime)):
            d2return = interpolate.griddata(coord1, d1[i,:,:].flatten(), coord2, method='linear')
            d2[i,0,:] = d2return
            d2[i,1,:] = d2return
            
        dat2[varn] = d2
    
        # Check nan values
        print("# of nan in interpolated {}: {}".format(varn, np.isnan(d2).sum()))
        
    return dat2

In [ ]:
# FASTER: crop met_data_warped_noleap before bilinear_interpolate

# Step 1: Get bounding box of gdf points
min_lon, max_lon = hillslope_gdf.lon.min(), hillslope_gdf.lon.max()
min_lat, max_lat = hillslope_gdf.lat.min(), hillslope_gdf.lat.max()

# Step 2: Since met_data_warped_noleap is an xarray Dataset, use coordinate-based selection
# Expand by a buffer for interpolation (roughly 3 pixels worth)
x_coords = met_data_warped_noleap.x.values
y_coords = met_data_warped_noleap.y.values

# Calculate pixel spacing
dx = x_coords[1] - x_coords[0] if len(x_coords) > 1 else 1000
dy = y_coords[1] - y_coords[0] if len(y_coords) > 1 else 1000

# Add buffer (3 pixels)
buffer = 3
x_buffer = abs(dx) * buffer
y_buffer = abs(dy) * buffer

# Calculate selection bounds
x_min_select = min_lon - x_buffer
x_max_select = max_lon + x_buffer
y_min_select = min_lat - y_buffer
y_max_select = max_lat + y_buffer

# For slice() to work correctly with xarray, we need to ensure the order matches the coordinate order
# If coordinates are descending, swap the min/max in the slice
x_slice = slice(x_min_select, x_max_select) if x_coords[0] < x_coords[-1] else slice(x_max_select, x_min_select)
y_slice = slice(y_min_select, y_max_select) if y_coords[0] < y_coords[-1] else slice(y_max_select, y_min_select)

# Select subset using xarray's sel method with buffer
subset_xr = met_data_warped_noleap.sel(x=x_slice, y=y_slice)

# Extract subset data as dictionary
subset_data = {var: subset_xr[var].values for var in subset_xr.data_vars}

# Step 3: Get the coordinate arrays for the subset
xs_1d = subset_xr.x.values
ys_1d = subset_xr.y.values

# Create 2D coordinate meshgrid
xs, ys = np.meshgrid(xs_1d, ys_1d)

# Step 4: Flatten for interpolation
xmesh_flatten = xs.flatten()
ymesh_flatten = ys.flatten()

raw_dat_2dtran = bilinear_interpolate(xmesh_flatten, 
                                      ymesh_flatten, 
                                      hillslope_gdf.lon.values, 
                                      hillslope_gdf.lat.values, 
                                      subset_data)

In [ ]:
raw_dat_2dtran['APCP_surface'].shape # dim=(time, y, x)

In [ ]:
# Plot spatial averaged rainfall
for varn in raw_dat_2dtran:
    fig,ax = plt.subplots(1,1,figsize=(15,3))
    ax.plot(raw_dat_2dtran[varn].mean(axis=(1,2)), '*-')
    ax.set(ylabel=varn)

In [ ]:
# noticing, raw_dat_2dtran is interpolated from met_data_warped_noleap, so the precipitation unit is [kg/m^2]
print("Maximum value: " + str(max(raw_dat_2dtran['APCP_surface'].mean(axis=(1,2)))))
print("Minimum value: " + str(min(raw_dat_2dtran['APCP_surface'].mean(axis=(1,2)))))
print("Sum of values: " + str(sum(raw_dat_2dtran['APCP_surface'].mean(axis=(1,2)))))
print("Average value: " + str(sum(raw_dat_2dtran['APCP_surface'].mean(axis=(1,2)))/raw_dat_2dtran['APCP_surface'].shape[0]))

In [ ]:
met_data_warped_noleap

In [ ]:
varn='APCP_surface'
fig,ax = plt.subplots(1,1,figsize=(8,3))

times = np.array([datetime.datetime(*t.timetuple()[0:6]) for t in met_data_warped_noleap['time'].values])

ax.plot(times, raw_dat_2dtran[varn].mean(axis=(1,2)), '*-')
ax.set(ylabel='APCP_surface [mm/d]')

## Generate the typical year DayMet for spinup

- note for spinup DayMet
    - in ww v1.5 Coweeta example, there are two approaches, average and median
    - in Bing's notebook, it just take the 1st year repeatly
    - in Zhi's notebook, it uses the averaged
- note for DayMet x and y
    - 

In [ ]:
# target x and y used in DayMet h5 file for 2D transect ATS
x_2dtran, y_2dtran = hillslope_gdf.h_distance.values.flatten(), np.array([0.0, 1.0])
print(x_2dtran)
print(y_2dtran)

In [ ]:
import copy

# In watershed-workflow 2.0, met_data_warped_noleap is an xarray Dataset
# We need to create a new Dataset with the interpolated data

# First, create coordinate arrays for the 2D transect
from affine import Affine
dx_2dtran = (x_2dtran[-1] - x_2dtran[0])/meshsize_nx #x_2dtran[1] - x_2dtran[0]
new_transform = Affine(dx_2dtran, 0, 0,  # (a, b, c)
                       0, -1, len(y_2dtran)-1)    # (d, e, f)

## verify the Affine here, related to watershed_workflow.io.write_dataset_to_hdf5 below
## the target x = 0:dx:680, y=[0,1]
## Affine defines the transformation from [col, row] to [x,y]
## x = a*col + b*row + c
## y = d*col + e*row + f
x = np.array([(new_transform * (i, 0))[0] for i in range(len(x_2dtran))])
y = np.array([(new_transform * (0, j))[1] for j in range(len(y_2dtran))])
print("x coordinates:", x)
print("y coordinates:", y) # [note] y will be reverted in watershed_workflow.io.write_dataset_to_hdf5

# Create a new xarray Dataset with the interpolated data
data_vars = {}
for var_name, var_data in raw_dat_2dtran.items():
    data_vars[var_name] = (['time', 'y', 'x'], var_data)

# Get time coordinates from met_data_warped_noleap
time_coords = met_data_warped_noleap['time'].values

raw_dat_2dtran_warped = xr.Dataset(
    data_vars=data_vars,
    coords={
        'time': time_coords,
        'y': y_2dtran,
        'x': x_2dtran
    }
)

# Copy attributes from original dataset if they exist
if hasattr(met_data_warped_noleap, 'attrs'):
    raw_dat_2dtran_warped.attrs = copy.deepcopy(met_data_warped_noleap.attrs)

In [ ]:
raw_dat_2dtran_warped

In [ ]:
bounds = [x_2dtran[0], y_2dtran[0], x_2dtran[-1], y_2dtran[-1]]
print(bounds)

In [ ]:
# In xarray, access time using .time.values instead of .times
time_values = raw_dat_2dtran_warped.time.values

cftime_startdate_spinup = cftime.DatetimeNoLeap(start_year_spinup, 1, 1, 0, 0, 0, 0, has_year_zero=True)
index_startdate_spinup  = (np.where(time_values == cftime_startdate_spinup))[0][0]
cftime_enddate_spinup   = cftime.DatetimeNoLeap(end_year_spinup, 12, 31, 0, 0, 0, 0, has_year_zero=True)
index_enddate_spinup    = (np.where(time_values == cftime_enddate_spinup))[0][0]

cftime_startdate_transient = cftime.DatetimeNoLeap(start_year_transient, 1, 1, 0, 0, 0, 0, has_year_zero=True)
index_startdate_transient  = (np.where(time_values == cftime_startdate_transient))[0][0]
cftime_enddate_transient   = cftime.DatetimeNoLeap(end_year_transient, 12, 31, 0, 0, 0, 0, has_year_zero=True)
index_enddate_transient    = (np.where(time_values == cftime_enddate_transient))[0][0]

print([index_startdate_transient, index_enddate_transient])
print([index_startdate_spinup, index_enddate_spinup])

In [ ]:
# split raw_dat_2dtran_warped to spinup and transient
# - raw_dat_2dtran_warped_spinup
# - raw_dat_2dtran_warped_transient

# In xarray, use isel() for index-based selection to slice by time
# spinup
raw_dat_2dtran_warped_spinup = raw_dat_2dtran_warped.isel(
    time=slice(index_startdate_spinup, index_enddate_spinup+1)
)

# transient
raw_dat_2dtran_warped_transient = raw_dat_2dtran_warped.isel(
    time=slice(index_startdate_transient, index_enddate_transient+1)
)

print(f"Spinup dataset shape: time={len(raw_dat_2dtran_warped_spinup.time)}")
print(f"Transient dataset shape: time={len(raw_dat_2dtran_warped_transient.time)}")

## Write to ATS/HDF5 format for transient run

In [ ]:
outputs['daymet_transient_filename_site'] = f'../data-processed/{site_name}/{site_name}_DayMet_{start_year_transient}_{end_year_transient}.h5'
# In watershed-workflow 2.0, for AORC data use convertAORCToATS instead of convertToATS
raw_dat_2dtran_warped_transient_ats = watershed_workflow.meteorology.convertAORCToATS(raw_dat_2dtran_warped_transient)
# For writing, we may need to create attrs differently for xarray datasets
# The getAttributes function should still work with bounds and dates
attrs_transient = {'bounds': bounds, 'start_date': startdate_transient, 'end_date': enddate_transient}
watershed_workflow.io.writeDatasetToHDF5(
    outputs['daymet_transient_filename_site'],
    raw_dat_2dtran_warped_transient_ats,
    attrs_transient)
#    raw_dat_2dtran_warped_transient_ats.attrs)

In [ ]:
raw_dat_2dtran_warped_transient_ats['time']

In [ ]:
# plot the transient DayMet h5 file for ATS
fig = plt.figure(figsize=(15,6))
ax = fig.add_subplot(221)

times = raw_dat_2dtran_warped_transient_ats['time']
prain_spatial_mean_raw = raw_dat_2dtran_warped_transient_ats['precipitation rain [m s^-1]'].data.mean(axis=(1,2))#.data[:,0,5]
psnow_spatial_mean_raw = raw_dat_2dtran_warped_transient_ats['precipitation snow [m SWE s^-1]'].data.mean(axis=(1,2))#.data[:,0,5]
ax.plot(times, prain_spatial_mean_raw, 'b', label='rain')
ax.plot(times, psnow_spatial_mean_raw, 'c', label='snow')
ax.set_ylabel('precip [m s^-1]')
ax.legend()

ax = fig.add_subplot(222)
qswin_spatial_mean_raw = raw_dat_2dtran_warped_transient_ats['incoming shortwave radiation [W m^-2]'].data.mean(axis=(1,2))#.data[:,0,5]
ax.plot(times, qswin_spatial_mean_raw, 'r')
ax.set_ylabel('incoming shortwave radiation [W m^-2]')

ax = fig.add_subplot(223)
airtemp_spatial_mean_raw = raw_dat_2dtran_warped_transient_ats['air temperature [K]'].data.mean(axis=(1,2))#.data[:,0,5]
ax.plot(times, airtemp_spatial_mean_raw, 'g')
ax.set_ylabel('air temperature [K]')

ax = fig.add_subplot(224)
vapor_pressure_spatial_mean_raw = raw_dat_2dtran_warped_transient_ats['vapor pressure air [Pa]'].data.mean(axis=(1,2))#.data[:,0,5]
ax.plot(times, vapor_pressure_spatial_mean_raw, 'm')
ax.set_ylabel('vapor pressure air [Pa]')
plt.show()

## Write to ATS/HDF5 format for cyclic spinup

This will write daymet in a format that ATS can read. E.g., this will partition precipitation into rain and snow, convert vapor pressure to relative humidity, get mean air temperature and so on.

- dout has dims of `(ntime, nrow, ncol)` or `(ntime, ny, nx)`, where nrow = ny = 1

In [ ]:
# compute the typical year of the _raw_ data
# note that we set interpolate to False, since met_data is already daily on a noleap calendar
# In watershed-workflow 2.0, computeAverageYear should work with xarray Datasets
dat_2dtran_spinup_typical = watershed_workflow.data.computeAverageYear(
    raw_dat_2dtran_warped_spinup,
    cftime.DatetimeNoLeap(2010, 1, 1),
    nyears_cyclic_spinup
)
# convert that to ATS
dat_2dtran_spinup_typical_ats = watershed_workflow.meteorology.convertAORCToATS(dat_2dtran_spinup_typical)

In [ ]:
# plot the smoothed precip result
fig = plt.figure(figsize=(15,3))
ax = fig.add_subplot(121)

#times = np.array([datetime.datetime(*t.timetuple()[0:6]) for t in dat_2dtran_spinup_smooth_ats.times])
times = dat_2dtran_spinup_typical_ats['time']
prain_spatial_mean_spinup = dat_2dtran_spinup_typical_ats['precipitation rain [m s^-1]'].data.mean(axis=(1,2))#.data[:,0,5]
psnow_spatial_mean_spinup = dat_2dtran_spinup_typical_ats['precipitation snow [m SWE s^-1]'].data.mean(axis=(1,2))#.data[:,0,5]
ax.plot(times, prain_spatial_mean_spinup, 'b', label='rain')
ax.plot(times, psnow_spatial_mean_spinup, 'c', label='snow')

ax.set_ylabel('precip [m s^-1]')
ax.legend()

ax = fig.add_subplot(122)
qswin_spatial_mean_spinup = dat_2dtran_spinup_typical_ats['incoming shortwave radiation [W m^-2]'].data.mean(axis=(1,2))#.data[:,0,5]
ax.plot(times, qswin_spatial_mean_spinup, 'r')
ax.set_ylabel('incoming shortwave radiation [W m^-2]')
plt.show()

In [ ]:
outputs['daymet_spinup_filename_site'] = f'../data-processed/{site_name}/{site_name}_DayMet_typical{nyears_cyclic_spinup}yr_{start_year_spinup}_{end_year_spinup}.h5'

attrs_spinup = {'bounds': bounds, 'start_date': startdate_spinup, 'end_date': enddate_spinup}
watershed_workflow.io.writeDatasetToHDF5(
    outputs['daymet_spinup_filename_site'],
    dat_2dtran_spinup_typical_ats,
    attrs_spinup)

In [ ]:
# Calculate the mean values (equivalent to your original calculation)
prain_spatial_temporal_mean = np.mean(prain_spatial_mean_spinup)
psnow_spatial_temporal_mean = np.mean(psnow_spatial_mean_spinup)
mean_precip4ats = prain_spatial_temporal_mean + psnow_spatial_temporal_mean

print(f'Mean annual precip rate [m s^-1] = {mean_precip4ats}')

## Merged data for ats-pflotran transient
- mainly for restart use

In [ ]:
outputs['daymet_merge_filename_site'] = f'../data-processed/{site_name}/{site_name}_DayMet_merged.h5'

In [ ]:
list(raw_dat_2dtran_warped_transient_ats.data_vars)

In [ ]:
# --- Merge spinup and transient Data ---
print(f"Merging DayMet data into:\n  {outputs['daymet_merge_filename_site']}")

with h5.File(outputs['daymet_merge_filename_site'], 'w') as hdf:
    # --- Time Handling (same logic as before) ---
    total_years = nyears_cyclic_spinup + (end_year_transient - start_year_transient + 1)
    total_days = total_years*365
    merged_time = np.arange(total_days) * 24 * 3600
    hdf.create_dataset('time [s]', data=merged_time)
    print("  - Merged 'time [s]'")
    
    # --- Copy Static Spatial Data ---
    hdf.create_dataset('x [m]', data=x_2dtran)
    hdf.create_dataset('y [m]', data=y_2dtran)
    print("  - Copied 'x [m]' and 'y [m]'")

    # --- Loop through all keys and merge each variable ---
    all_keys = list(raw_dat_2dtran_warped_transient_ats.data_vars)
    
    for key in all_keys:
        print(f"\n  - Processing '{key}'...")
        
        # Get the spinup and transient arrays for this key
        spinup_array = dat_2dtran_spinup_typical_ats[key]
        transient_array = raw_dat_2dtran_warped_transient_ats[key]
        
        print(f"    Spinup shape: {spinup_array.shape}")
        print(f"    Transient shape: {transient_array.shape}")
        
        # Concatenate along the time axis (axis=0)
        merged_array = np.concatenate((spinup_array, transient_array), axis=0)
        
        print(f"    Merged shape: {merged_array.shape}")
        
        # Create a group for this variable
        var_group = hdf.create_group(key)
        
        # Save each time step as a separate dataset
        for i in range(merged_array.shape[0]):
            var_group.create_dataset(str(i), data=merged_array[i])
        
        print(f"    ✓ Saved '{key}' as a group with {merged_array.shape[0]} time steps")

print("\n✓ DayMet merge complete.")
print(f"✓ All {len(all_keys)} variables merged successfully.")

In [ ]:
outputs